## Data Exploring

* Input file: Historical Product Demand.csv 


* Description: CSV data file containing product demand for encoded product id's


* Size of Data: (1048575, 5)


* Features: Product_Code, Warehouse, Product_Category, Date, Order_Demand


* Period: 2011-01-08 ~ 2017-01-09

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
df = pd.read_csv('Historical Product Demand.csv')

In [28]:
df.shape

(1048575, 5)

In [29]:
df.columns

Index(['Product_Code', 'Warehouse', 'Product_Category', 'Date',
       'Order_Demand'],
      dtype='object')

In [30]:
df.head()

,Product_Code,Warehouse,Product_Category,Date,Order_Demand
0,Product_0993,Whse_J,Category_028,2012/7/27,100
1,Product_0979,Whse_J,Category_028,2012/1/19,500
2,Product_0979,Whse_J,Category_028,2012/2/3,500
3,Product_0979,Whse_J,Category_028,2012/2/9,500
4,Product_0979,Whse_J,Category_028,2012/3/2,500


In [31]:
df.isna().sum()

Product_Code            0
Warehouse               0
Product_Category        0
Date                11239
Order_Demand            0
dtype: int64

In [32]:
#Drop na's.

# 결측치 행 1% 미만이라, drop 
df.dropna(axis=0, inplace=True) #remove all rows with na's.
df.reset_index(drop=True)
df.sort_values('Date')[10:20] #Some of the values have () in them.

,Product_Code,Warehouse,Product_Category,Date,Order_Demand
44795,Product_0965,Whse_A,Category_006,2011/11/18,1
44796,Product_0965,Whse_A,Category_006,2011/11/21,3
44797,Product_0965,Whse_A,Category_006,2011/11/21,5
44798,Product_0965,Whse_A,Category_006,2011/11/21,2
119561,Product_0980,Whse_A,Category_028,2011/11/21,100
107158,Product_0138,Whse_J,Category_007,2011/11/22,188
107159,Product_0138,Whse_J,Category_007,2011/11/22,1852
111727,Product_0982,Whse_A,Category_028,2011/11/22,3700
44102,Product_0980,Whse_A,Category_028,2011/11/23,1000
71915,Product_0980,Whse_A,Category_028,2011/11/23,200


In [33]:
print("The Number of unique")
print('-----------------------------')
print('Product code:\t', df.Product_Code.nunique())
print('Category:\t', df.Product_Category.nunique())
print('Warehouse:\t', df.Warehouse.nunique())

The Number of unique
-----------------------------
Product code:	 2160
Category:	 33
Warehouse:	 4


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1037336 entries, 0 to 1048574
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   Product_Code      1037336 non-null  object
 1   Warehouse         1037336 non-null  object
 2   Product_Category  1037336 non-null  object
 3   Date              1037336 non-null  object
 4   Order_Demand      1037336 non-null  object
dtypes: object(5)
memory usage: 47.5+ MB


In [35]:
#Target Feature - Order_Demand
# 종속 변수에 () 가 포함되어 있어, 모두 제거하고 int로 변환
df['Order_Demand'] = df['Order_Demand'].str.replace('(',"")
df['Order_Demand'] = df['Order_Demand'].str.replace(')',"")

df['Order_Demand'] = df['Order_Demand'].astype('int64')

C:\Users\7info\AppData\Local\Temp\ipykernel_33396\3703683769.py:3: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df['Order_Demand'] = df['Order_Demand'].str.replace('(',"")
C:\Users\7info\AppData\Local\Temp\ipykernel_33396\3703683769.py:4: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df['Order_Demand'] = df['Order_Demand'].str.replace(')',"")


In [36]:
# datetime dtype으로 변환
from datetime import datetime

df['Date'] = df['Date'].apply(lambda x: datetime.strptime(x, '%Y/%m/%d'))

### 모든 상품을 하나의 판매량으로 그룹화

In [37]:
df_sum = df.groupby('Date')['Order_Demand'].sum().reset_index()
df_sum = df_sum.set_index('Date')
df_sum

,Order_Demand
Date,
2011-01-08,2
2011-05-31,108
2011-06-24,92000
2011-09-02,1250
2011-09-27,28
...,...
2017-01-03,2400
2017-01-04,29250
2017-01-05,83929


In [41]:
# train: 2011 ~ 2015 (81%)
# test: 2015 ~ 2017.1 (19%)
train_df = df_sum[(df_sum.index <='2015-12-31')].sort_values('Date', ascending=True)
test_df = df_sum[(df_sum.index > '2015-12-31')].sort_values('Date', ascending=True) 

## Time Series Graph